# TriageGrid training and evaluation
Train only on the supplied synthetic dataset. The CSV and JSONL contain the same cases. Duplicate input strings are grouped in cross-validation. These results do not establish clinical safety.

In [ ]:
from pathlib import Path
import os, sys, runpy, json
root = Path.cwd() if (Path.cwd()/"backend").exists() else Path.cwd().parent
os.chdir(root)
sys.path.insert(0, str(root/"backend"))
import pandas as pd
df = pd.read_csv("data/provided/triage_dataset_flat.csv")
display(df.groupby(["care_type", "urgency"], dropna=False).size().rename("count"))
print("Rows:", len(df), "Unique inputs:", df.user_input.str.strip().str.casefold().nunique())

In [ ]:
# Explicit fit and grouped baseline evaluation. Saves both complete TF-IDF pipelines.
runpy.run_path("scripts/train_models.py", run_name="__main__")

In [ ]:
meta=json.loads(Path("models/baseline/model_meta.json").read_text())
display(pd.DataFrame(meta["care_type_report"]).T)
display(pd.DataFrame(meta["urgency_report"]).T)

## Integration
The API and CSV CLI use `triagegrid.services.engine.predict`. Caregiver selection uses `caregivers.csv` and is not learned from the classifier. Run `pytest backend/tests` to check source conversations, safety, booking ownership, concurrent slot conflicts and consent. No real patient data is permitted.